## 调试 Au2-MgO-Al 能量指标（energy metric）

目的：
- 检查从 `Au-MgO-Al.xyz` 读入后，batch 中有哪些键；
- 确认能量究竟存在哪个键（`energy` 还是别的）；
- 在一个小 batch 上手动跑一次 `Metrics` 逻辑，看为什么会报 `KeyError: 'energy'`。

In [5]:
import os, sys
from pathlib import Path

import torch

ROOT_DIR = Path(os.getcwd()).resolve()
if ROOT_DIR.name != "fit-4hdnnp-Au2-MgO-Al":
    ROOT_DIR = ROOT_DIR / "fit-4hdnnp-Au2-MgO-Al"
PROJECT_ROOT = ROOT_DIR.parent  # CACE-SOG-Qeq
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import cace
from cace.tasks import get_dataset_from_xyz, load_data_loader
from cace.tools import Metrics

print("PROJECT_ROOT:", PROJECT_ROOT)
print("WORKING DIR:", os.getcwd())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

PROJECT_ROOT: /work/home/acrb3qk4vo/SOG-Qeq/SOG-Net/CACE-SOG-Qeq
WORKING DIR: /work/home/acrb3qk4vo/SOG-Qeq/SOG-Net/CACE-SOG-Qeq/fit-4hdnnp-Au2-MgO-Al
device: cpu


In [6]:
# 读取数据集，参数与 fit-cace-SOG.py 保持一致
cutoff = 5.0
xyz_path = PROJECT_ROOT / "fit-4hdnnp-Au2-MgO-Al" / "Au-MgO-Al.xyz"
print("xyz_path exists:", xyz_path.exists())

collection = get_dataset_from_xyz(
    train_path=str(xyz_path),
    cutoff=cutoff,
    valid_fraction=0.1,
    seed=1,
    data_key={"energy": "energy", "forces": "forces"},
    atomic_energies={
        8: -18599.43617104475,
        12: -8721.75974245582,
        13: -9877.676428588728,
        79: -688.8680063349827,
    },
)

train_loader = load_data_loader(collection, data_type="train", batch_size=4)
print("#train structures:", len(collection.train))
print("#valid structures:", len(collection.valid))

xyz_path exists: True
#train structures: 4500
#valid structures: 500


In [7]:
# 取一个 batch，看有哪些键，以及 energy / forces 的形状
batch = next(iter(train_loader))
print("type(batch):", type(batch))
print("batch keys:", batch.keys)

batch_dict = batch.to_dict()
print("batch_dict keys:", list(batch_dict.keys()))

for k in ["energy", "forces", "CACE_energy", "CACE_forces"]:
    v = batch_dict.get(k, None)
    if v is None:
        print(f"{k}: None")
    else:
        print(f"{k}: shape={tuple(v.shape)}, dtype={v.dtype}")

type(batch): <class 'cace.tools.torch_geometric.batch.Batch'>
batch keys: ['edge_index', 'batch', 'ptr', 'positions', 'shifts', 'unit_shifts', 'cell', 'atomic_numbers']
batch_dict keys: ['atomic_numbers', 'batch', 'cell', 'edge_index', 'positions', 'ptr', 'shifts', 'unit_shifts']
energy: None
forces: None
CACE_energy: None
CACE_forces: None


In [ ]:
# 尝试用 NaCl 那个 extxyz 读取函数直接读 Au2-MgO-Al（自定义 z_map）

from cace.data.extxyz_charge import read_extxyz_with_charge

z_map_au = {"O": 8, "Mg": 12, "Al": 13, "Au": 79}

extxyz_data = read_extxyz_with_charge(
    path=str(xyz_path),
    cutoff=cutoff,
    atomic_energies={
        8: -18599.43617104475,
        12: -8721.75974245582,
        13: -9877.676428588728,
        79: -688.8680063349827,
    },
    z_map=z_map_au,  # 关键是这行
)

print("len(extxyz_data) =", len(extxyz_data))
first = extxyz_data[0]
print("keys of first AtomicData:", first.keys)
print("energy (per-structure):", first["energy"])
print("forces shape:", None if first["forces"] is None else tuple(first["forces"].shape))
print("charge shape:", None if first["charge"] is None else tuple(first["charge"].shape))
print("atomic_numbers unique:", torch.unique(first["atomic_numbers"]))

KeyError: 'Mg'

In [9]:
from itertools import islice

with open(xyz_path, "r") as f:
    # 跳过第一帧的两行头，再看接下来的十几行
    n = int(f.readline().strip())
    header = f.readline().strip()
    print("header:", header)
    species_seen = set()
    for line in islice(f, 0, n):
        parts = line.split()
        if parts:
            species_seen.add(parts[0])
print("unique species in first frame:", species_seen)

header: Lattice="9.047430736655434 0.0 0.0 0.0 9.047430736655434 0.0 0.0 0.0 26.458860890475076" Properties=species:S:1:pos:R:3:forces:R:3:charge:R:1 simulation_info=input energy=-1480190.056621997 pbc="T T T"
unique species in first frame: {'Al', 'Au', 'Mg', 'O'}
